## Pure-CO2 interfacial tension: PC-SAFT (cDFT) vs REFPROP
Reference benchmark plot for the mixture sensitivity analysis. Computes
- REFPROP saturation γ, ρ_l, ρ_v on a coarse T grid (via CoolProp's REFPROP backend)
- PC-SAFT cDFT γ, ρ_l, ρ_v, L90/10 thickness on the same coarse grid (with a dynamic-grid retry on convergence failure)
- A continuous PC-SAFT γ(T) curve via `SurfaceTensionDiagram` for plotting
and overlays them with critical points marked.

In [ ]:
import os, sys, time
from pathlib import Path

THERMOIFT_SRC = Path('../thermoift/src').resolve()
if str(THERMOIFT_SRC) not in sys.path:
    sys.path.insert(0, str(THERMOIFT_SRC))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator

import si_units as si
import CoolProp.CoolProp as CP
import feos
from feos import (Parameters, HelmholtzEnergyFunctional, State, PhaseEquilibrium,
                  PlanarInterface, PhaseDiagram, SurfaceTensionDiagram)
import thermoift
from thermoift import PLOT_SETTINGS as ps

notebook_start = time.perf_counter()

## Parameter cell

In [ ]:
FLUID         = 'CO2'
REFPROP_PATH  = os.path.expanduser('~/Software/REFPROP/REFPROP-cmake/build')
PARAMS_JSON   = str(Path(thermoift.__file__).parent / 'parameters.json')

T_array       = np.arange(200, 305, 1)   # coarse grid for tabulated comparison
T_curve_min   = 200                       # start T for the continuous PC-SAFT curve
N_CURVE       = 200                       # # phase-diagram points for the continuous curve

N_GRID        = 1024                       # cDFT grid points (base)
L_GRID_A      = 100                       # cDFT domain length [Angstrom] (base)
N_GRID_RETRY  = 256                       # on convergence failure: coarser grid
L_GRID_RETRY  = 500                       # on convergence failure: wider domain

CSV_REFPROP    = 'CO2_REFPROP.csv'
CSV_PCSAFT     = 'CO2_pcsaft.csv'
CSV_PCSAFT_SAT = 'CO2_pcsaft_saturation.csv'   # T, P, gamma along the pure-CO2 PC-SAFT saturation line
PLOT_NAME      = 'PC-SAFT_IFT_CO2'             # ps.save_plot writes PNG + PDF into PLOTS/

## REFPROP reference

In [ ]:
CP.set_config_string(CP.ALTERNATIVE_REFPROP_PATH, REFPROP_PATH)
molar_mass_kg_per_mol = CP.PropsSI('M', FLUID)           # kg/mol
molar_mass_g_per_mol  = molar_mass_kg_per_mol * 1000     # g/mol (script's variable)
Tc_RP = CP.PropsSI('TCRIT', FLUID)
print(f'Molar mass of {FLUID}: {molar_mass_g_per_mol:.3f} g/mol')
print(f'Critical temperature of {FLUID} from REFPROP: {Tc_RP:.2f} K')
print(f'T_array: {T_array}')

ift_REFPROP, rl_REFPROP, rv_REFPROP = [], [], []
with open(CSV_REFPROP, 'w') as f:
    f.write('T(K),rho_l(kg/m3),rho_v(kg/m3),sigma(mN/m)\n')
    for T in T_array:
        try:
            sigma = CP.PropsSI('I', 'T', T, 'Q', 1, FLUID)           # N/m
            rho_l = CP.PropsSI('D', 'T', T, 'Q', 0, FLUID)           # kg/m3
            rho_v = CP.PropsSI('D', 'T', T, 'Q', 1, FLUID)
            ift_REFPROP.append(round(sigma * 1000, 6))
            rl_REFPROP.append(round(rho_l, 6))
            rv_REFPROP.append(round(rho_v, 6))
            f.write(f'{T:.3f},{rho_l:.4f},{rho_v:.4f},{sigma*1000:.4f}\n')
        except Exception as e:
            print(f'  REFPROP failed at T={T} K: {e}')
            ift_REFPROP.append(np.nan); rl_REFPROP.append(np.nan); rv_REFPROP.append(np.nan)

print(f'Wrote {CSV_REFPROP}')

## PC-SAFT cDFT solve on the same T_array

In [ ]:
parameters = Parameters.from_json(['carbon dioxide'], PARAMS_JSON)
pcsaft     = HelmholtzEnergyFunctional.pcsaft(parameters)
cp_state   = State.critical_point(pcsaft)
Tc_PC      = float(cp_state.temperature / si.KELVIN)
print(f'Critical temperature of {FLUID} from FEOS: {Tc_PC:.2f} K')

In [ ]:
def _solve_planar(T, length_A, n_grid):
    """Run one cDFT planar-interface solve at T. Returns (rho_l, rho_v, gamma_mN_m, delta_A)."""
    vle = PhaseEquilibrium.pure(pcsaft, T * si.KELVIN)
    interface = PlanarInterface.from_tanh(
        vle=vle, n_grid=n_grid, l_grid=length_A * si.ANGSTROM,
        critical_temperature=cp_state.temperature,
    )
    surface_tension = interface.solve().surface_tension
    rho_l = float((vle.liquid.density * molar_mass_g_per_mol) / (si.KILO * si.MOL / si.METER**3))
    rho_v = float((vle.vapor.density  * molar_mass_g_per_mol) / (si.KILO * si.MOL / si.METER**3))
    gamma = float(surface_tension * si.KILO / (si.NEWTON / si.METER))
    # 90/10 thickness: distance between density-profile points crossing 0.9 and 0.1 of the jump
    rho_profile = (interface.density / (si.KILO * si.MOL / si.METER**3) * molar_mass_g_per_mol)[0]
    nbin = int(rho_profile.shape[0])
    z    = np.linspace(0.005, 0.995, nbin) * length_A
    rho_start = rho_l + 0.9 * (rho_v - rho_l)   # 90% of way to vapor side
    rho_end   = rho_l + 0.1 * (rho_v - rho_l)   # 10% of way to vapor side
    i_start = int(np.abs(rho_profile[:nbin] - rho_start).argmin())
    i_end   = int(np.abs(rho_profile[:nbin] - rho_end  ).argmin())
    delta = float(z[i_start] - z[i_end])
    return rho_l, rho_v, gamma, delta

with open(CSV_PCSAFT, 'w') as f:
    f.write('T(K),rho_l(kg/m3),rho_v(kg/m3),sigma(mN/m),delta(A)\n')
    for T in T_array:
        try:
            rho_l, rho_v, gamma, delta = _solve_planar(T, L_GRID_A, N_GRID)
        except RuntimeError:
            print(f'[Warning] DFT did not converge at T={T:.2f} K. Retrying with l_grid={L_GRID_RETRY} A, n_grid={N_GRID_RETRY}.')
            try:
                rho_l, rho_v, gamma, delta = _solve_planar(T, L_GRID_RETRY, N_GRID_RETRY)
            except RuntimeError:
                print(f'[Fail] DFT still did not converge at T={T:.2f} K. Skipping.')
                continue
        f.write(f'{T:.3f},{rho_l:.4f},{rho_v:.4f},{gamma:.4f},{delta:.4f}\n')

print(f'Wrote {CSV_PCSAFT}')

## Continuous PC-SAFT γ(T) curve via SurfaceTensionDiagram

In [ ]:
vles = PhaseDiagram.pure(pcsaft, T_curve_min * si.KELVIN, N_CURVE,
                         critical_temperature=cp_state.temperature)
sfts = SurfaceTensionDiagram(vles.states, n_grid=N_GRID, l_grid=L_GRID_A * si.ANGSTROM,
                             critical_temperature=cp_state.temperature)

T_K     = np.asarray(sfts.liquid.temperature / si.KELVIN)
P_bar   = np.asarray(vles.liquid.pressure   / si.BAR)
gamma_m = np.asarray(sfts.surface_tension * 1000 / si.NEWTON * si.METER)

dft_data = pd.DataFrame({
    'Temperature (K)':         T_K,
    'Surf. Tension (mN/m)':    gamma_m,
})

# Saturation curve CSV — consumed by SENSITIVITY_ANALYSIS_HIRES.ipynb to overlay the
# pure-CO2 PC-SAFT reference on the PT phase map. Last row is the critical point.
sat_df = pd.DataFrame({'T_K': T_K, 'P_bar': P_bar, 'gamma_mN_m': gamma_m})
sat_df.to_csv(CSV_PCSAFT_SAT, index=False, float_format='%.6f')
print(f'Wrote {CSV_PCSAFT_SAT}  ({len(sat_df)} rows, Tc≈{T_K[-1]:.3f} K, Pc≈{P_bar[-1]:.3f} bar)')
print(dft_data.head())

## Comparison plot

In [ ]:
fig, ax = ps.plot_init()

ax.plot(dft_data['Temperature (K)'], dft_data['Surf. Tension (mN/m)'],
        label='PC-SAFT', linestyle='solid', linewidth=ps.linewidth, color=ps.colors[0])
ax.plot(T_array, ift_REFPROP,
        label='REFPROP', linestyle='solid', linewidth=ps.linewidth, color=ps.colors[1])

ax.scatter([Tc_PC], [0.0], clip_on=False, color='k', label='Critical point')
ax.scatter([Tc_RP], [0.0], clip_on=False, color='k')

ax.set_ylabel(r'$\gamma$ $/$ $[\mathrm{mN\,m^{-1}}]$', fontsize=12)
ax.set_xlabel(r'$T$ $/$ $[\mathrm{K}]$',               fontsize=12)
ax.yaxis.set_major_locator(MultipleLocator(3))
ax.yaxis.set_minor_locator(MultipleLocator(1.5))
ax.xaxis.set_major_locator(MultipleLocator(20))
ax.xaxis.set_minor_locator(MultipleLocator(10))

ax.legend(fontsize=ps.legend_fontsize, loc='upper right', ncol=1,
          borderaxespad=ps.borderaxespad)
plt.tight_layout()
ps.save_plot(fig, PLOT_NAME)   # writes PLOTS/<PLOT_NAME>.{png,pdf}
plt.show()

print(f'Total execution time: {time.perf_counter() - notebook_start:.2f} seconds')